In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [ ]:
test_data.data.shape

In [ ]:
batch_size = 64

In [ ]:
# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [ ]:
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [ ]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)

In [ ]:
# Optimizing the Model Parameters
# ----------------------------------------
# To train a model, we need a `loss function <https://pytorch.org/docs/stable/nn.html#loss-functions>`_
# and an `optimizer <https://pytorch.org/docs/stable/optim.html>`_.

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [ ]:
#######################################################################
# In a single training loop, the model makes predictions on the training dataset (fed to it in batches), and
# backpropagates the prediction error to adjust the model's parameters.

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
##############################################################################
# We also check the model's performance against the test dataset to ensure it is learning.

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
##############################################################################
# The training process is conducted over several iterations (*epochs*). During each epoch, the model learns
# parameters to make better predictions. We print the model's accuracy and loss at each epoch; we'd like to see the
# accuracy increase and the loss decrease with every epoch.

epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

In [ ]:
######################################################################
# Saving Models
# -------------
# A common way to save a model is to serialize the internal state dictionary (containing the model parameters).

torch.save(model.state_dict(), "/home/ts/model.pth")
print("Saved PyTorch Model State to model.pth")

In [ ]:
######################################################################
# Loading Models
# ----------------------------
#
# The process for loading a model includes re-creating the model structure and loading
# the state dictionary into it.

model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("/home/ts/model.pth", weights_only=True))

#############################################################
# This model can now be used to make predictions.

classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len,  d_model)
        position = torch.arange(0,  max_len, dtype=torch.float).unsqueeze(1) 
        div_term = torch.exp(torch.arange(0,  d_model, 2).float() * (-math.log(10000.0)/d_model)) 
        pe[:, 0::2] = torch.sin(position  * div_term)
        pe[:, 1::2] = torch.cos(position  * div_term)
        self.register_buffer('pe',  pe.unsqueeze(0))   # [1, max_len, d_model]
 
    def forward(self, x):
        x = x + self.pe[:,  :x.size(1),  :]
        return x
 
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, nhead=8, 
                 num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048):
        super().__init__()
        # 输入嵌入层 
        self.src_embed  = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed  = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoder  = PositionalEncoding(d_model)
        
        # Transformer主体
        self.transformer  = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward
        )
        
        # 输出层 
        self.fc_out  = nn.Linear(d_model, tgt_vocab_size)
 
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, d_model=512):
        # 嵌入与位置编码 
        src = self.pos_encoder(self.src_embed(src)  * math.sqrt(d_model)) 
        tgt = self.pos_encoder(self.tgt_embed(tgt)  * math.sqrt(d_model)) 
        
        # 调整维度：[seq_len, batch_size, d_model]
        src = src.permute(1,  0, 2)
        tgt = tgt.permute(1,  0, 2)
        
        # 生成序列掩码 
        if tgt_mask is None:
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(len(tgt)).to(tgt.device) 
        
        # Transformer计算 
        output = self.transformer( 
            src, tgt, 
            src_key_padding_mask=src_mask,
            tgt_key_padding_mask=tgt_mask,
            memory_key_padding_mask=src_mask
        )
        
        # 输出层 
        return self.fc_out(output.permute(1,  0, 2))

In [ ]:
batch_size = 60
src_seq_len = 15  # 源语言序列长度
tgt_seq_len = 61 # 目标语言序列长度 
src_vocab = 5000  # 源语言词表大小 
tgt_vocab = 6000  # 目标语言词表大小 
 
# 生成虚拟数据（实际应使用真实分词数据）
src_data = torch.randint(0,  src_vocab, (batch_size, src_seq_len))
tgt_data = torch.randint(0,  tgt_vocab, (batch_size, tgt_seq_len))

In [ ]:
device = torch.device("cuda"  if torch.cuda.is_available()  else "cpu")
 
model = Transformer(src_vocab, tgt_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters(),  lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 假设0为填充符
 
for epoch in range(10):
    optimizer.zero_grad() 
    output = model(src_data.to(device),  tgt_data[:, :-1].to(device))
    
    # 计算损失（忽略最后一个预测位置）
    loss = criterion(
        output.reshape(-1,  tgt_vocab), 
        tgt_data[:, 1:].contiguous().view(-1).to(device)
    )
    
    loss.backward() 
    optimizer.step() 
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}") 